# ML Horse-Race Asset Pricing Pipeline
**Replicating & Extending Weigert et al. (RFS 2023)**

- Linear: LASSO, Elastic Net, PCR, PLS
- Nonlinear: Random Forest, GBM, XGBoost, LightGBM
- Strict rolling-window OOS validation
- Decile long-short portfolios
- SHAP explainability

In [1]:
import os, sys
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.models import AssetPricingModels
from src.oos_validation import OOSValidator
from src.portfolio import PortfolioConstructor
from src.shap_explain import SHAPExplainer

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('All modules imported successfully!')

All modules imported successfully!


/home/user/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
models = AssetPricingModels(random_state=42)

lasso = models.get_model('lasso')
xgb_model = models.get_model('xgboost')

available = ['lasso', 'elasticnet', 'pcr', 'pls',
             'randomforest', 'gbm', 'xgboost', 'lightgbm',
             'nn', 'ensemble']
print('Available models:', available)

Available models: ['lasso', 'elasticnet', 'pcr', 'pls', 'randomforest', 'gbm', 'xgboost', 'lightgbm', 'nn', 'ensemble']


In [3]:
# Load data
ff5 = pd.read_csv('/home/user/Downloads/ml-asset-pricing-horse-rac-main/data/ff5_factors.csv', index_col=0, parse_dates=True)
print(f'FF5 data loaded: {ff5.shape}')
ff5.head()

FF5 data loaded: (15290, 6)


,Mkt-RF,SMB,HML,RMW,CMA,RF
Date,,,,,,
1963-07-01,-0.67,0.02,-0.35,0.03,0.13,0.012
1963-07-02,0.79,-0.28,0.28,-0.08,-0.21,0.012
1963-07-03,0.63,-0.18,-0.10,0.13,-0.25,0.012
1963-07-05,0.40,0.09,-0.28,0.07,-0.30,0.012
1963-07-08,-0.63,0.07,-0.20,-0.27,0.06,0.012


In [4]:
# Prepare demo data
data = ff5.copy()
data['target'] = data['Mkt-RF'].shift(-1)
data = data.dropna()

X = data.drop(columns=['target'])
y = data['target']
print(f'Features: {X.shape[1]} | Samples: {len(X)}')

Features: 6 | Samples: 15289


In [5]:
# Initialize components
validator = OOSValidator(n_splits=5, test_size=252)
portfolio = PortfolioConstructor(n_quantiles=10, periods_per_year=252)
explainer = SHAPExplainer(output_dir='../output/shap')
print('Components initialized')

Components initialized


In [6]:
# Run horse-race
# NOTE: adjust keys below to match output of cell b2
model_dict = {
    'LASSO':        'lasso',
    'ElasticNet':   'elastic_net',
    'PCR':          'pcr',
    'PLS':          'pls',
    'RandomForest': 'random_forest',
    'GBM':          'gbm',
    'XGBoost':      'xgboost',
    'LightGBM':     'lightgbm'
}

results = {}
for display_name, key in model_dict.items():
    print(f'Training {display_name} ...')
    model = models.get_model(key)
    oos_preds = []
    oos_actuals = []
    for fold in validator.split(X, y):
        X_train, X_test, y_train, y_test = fold
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        oos_preds.extend(preds)
        oos_actuals.extend(y_test.values)
    results[display_name] = {
        'preds': np.array(oos_preds),
        'actuals': np.array(oos_actuals)
    }

print('All models trained!')

Training LASSO ...
Training ElasticNet ...
Training PCR ...
Training PLS ...
Training RandomForest ...
Training GBM ...
Training XGBoost ...
Training LightGBM ...
All models trained!


In [11]:
perf_summary = {}
for name, res in results.items():
    idx = y.index[-len(res['preds']):]
    pred_series = pd.Series(res['preds'], index=idx)
    actual_series = pd.Series(res['actuals'], index=idx)
    port_df, ls_returns = portfolio.long_short_portfolios(pred_series, actual_series)
    perf = portfolio.evaluate_performance(ls_returns)
    perf_summary[name] = perf

summary = pd.DataFrame(perf_summary).T
print('Horse-Race Performance Summary (Long-Short Portfolio)')
summary.round(4)

Horse-Race Performance Summary (Long-Short Portfolio)


,mean_return,volatility,sharpe_ratio,max_drawdown
LASSO,-1.260,13.3624,-0.0943,-238.4400
ElasticNet,-1.380,13.3596,-0.1033,-238.4400
PCR,-2.272,13.3502,-0.1702,-238.4400
PLS,-2.272,13.3502,-0.1702,-238.4400
RandomForest,4.496,13.9663,0.3219,-197687.6500
GBM,6.996,14.0445,0.4981,-40.3098
XGBoost,6.026,14.2473,0.4230,-14.7907
LightGBM,0.374,14.0789,0.0266,-388428.0460


In [12]:
from pathlib import Path
explainer = SHAPExplainer(output_dir=str(Path.cwd() / "output" / "shap"))

# SHAP Explainability (XGBoost)
print('Generating SHAP explanations for XGBoost...')
model_xgb = models.get_model('xgboost')
model_xgb.fit(X.iloc[:-252], y.iloc[:-252])

result = explainer.explain_model(
    model=model_xgb,
    X_train=X.iloc[:-252],
    X_test=X.iloc[-252:],
    model_name='XGBoost'
)

print(f"Summary plot : {result['summary_path']}")  
print(f"Bar plot     : {result['bar_path']}")      
result['importance'].head(10)

Generating SHAP explanations for XGBoost...
Summary plot : /home/user/Downloads/ml-asset-pricing-horse-rac-main/output/shap/shap_summary_xgboost.png
Bar plot     : /home/user/Downloads/ml-asset-pricing-horse-rac-main/output/shap/shap_bar_xgboost.png


,feature,mean_abs_shap
0,Mkt-RF,0.075178
1,CMA,0.061480
2,HML,0.056401
3,RMW,0.050143
4,SMB,0.047375
5,RF,0.026741
